## Part A: Pre-Trained Summarization with T5 (Quick Demo)

In [ ]:
!pip install -q transformers datasets nltk rouge-score sentencepiece

from transformers import T5ForConditionalGeneration, T5Tokenizer
import textwrap


t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")

article = """
Artificial intelligence has made remarkable strides in recent years, transforming industries
from healthcare to finance. Machine learning models can now diagnose diseases from medical
images with accuracy rivaling human specialists. In the financial sector, AI algorithms
detect fraudulent transactions in real time, saving billions of dollars annually. Natural
language processing has enabled chatbots and virtual assistants that can understand and
respond to human queries with unprecedented fluency. However, these advances also raise
concerns about job displacement, algorithmic bias, and the ethical implications of
autonomous decision-making systems. Researchers and policymakers are working together
to develop frameworks that ensure AI is developed and deployed responsibly.
"""

# T5 expects the task prefix "summarize: " before the input
input_text = "summarize: " + article.strip()
input_ids = t5_tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)

summary_ids = t5_model.generate(
    input_ids,
    max_length=80,
    min_length=20,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary_text = t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("=" * 70)
print("ARTICLE:")
print(textwrap.fill(article.strip(), width=70))
print("\n" + "=" * 70)
print("T5 SUMMARY:")
print(textwrap.fill(summary_text, width=70))

In [ ]:
# Evaluate with ROUGE
from rouge_score import rouge_scorer

reference_summary = (
    "AI has transformed healthcare and finance but raises concerns "
    "about bias and job displacement."
)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference_summary, summary_text)

print("ROUGE Scores (T5 vs reference):")
for key, value in scores.items():
    print(f"  {key}: Precision={value.precision:.3f}  Recall={value.recall:.3f}  F1={value.fmeasure:.3f}")

---
## Part B: Transformer from Scratch for Summarization

We train a small encoder-decoder Transformer on a subset of XSum to learn abstractive summarization.

In [ ]:
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load XSum — take 2000 examples for speed
print("Loading XSum dataset...")
data = load_dataset("EdinburghNLP/xsum", split="train[:2000]", trust_remote_code=True)

# Prepare source (document) and target (summary) with <sos>/<eos>
documents = [x['document'][:300] for x in data]       # truncate long docs
summaries = [f"<sos> {x['summary'][:80]} <eos>" for x in data]  # truncate summaries

print(f"Loaded {len(documents)} document-summary pairs")
print(f"\nExample document (first 200 chars):\n  {documents[0][:200]}...")
print(f"\nExample summary:\n  {summaries[0]}")

In [ ]:
# Tokenize
VOCAB_SIZE = 8000
MAX_DOC_LEN = 80
MAX_SUM_LEN = 25

def tokenize(sentences, num_words=VOCAB_SIZE, maxlen=None):
    tok = Tokenizer(num_words=num_words, oov_token='<OOV>',
                    filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
    tok.fit_on_texts(sentences)
    seqs = tok.texts_to_sequences(sentences)
    padded = pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')
    return tok, padded

doc_tokenizer, doc_tensor = tokenize(documents, maxlen=MAX_DOC_LEN)
sum_tokenizer, sum_tensor = tokenize(summaries, maxlen=MAX_SUM_LEN)

doc_vocab_size = min(VOCAB_SIZE, len(doc_tokenizer.word_index) + 1)
sum_vocab_size = min(VOCAB_SIZE, len(sum_tokenizer.word_index) + 1)

print(f"Document vocab: {doc_vocab_size}, tensor shape: {doc_tensor.shape}")
print(f"Summary vocab:  {sum_vocab_size}, tensor shape: {sum_tensor.shape}")
print(f"<sos> ID: {sum_tokenizer.word_index.get('<sos>')}, <eos> ID: {sum_tokenizer.word_index.get('<eos>')}")

In [ ]:
# Build Transformer

class SummarizationTransformer(tf.keras.Model):
    def __init__(self, src_vocab, tgt_vocab, d_model=128, num_heads=4,
                 dff=256, max_src=80, max_tgt=25, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        self.enc_emb = tf.keras.layers.Embedding(src_vocab, d_model)
        self.dec_emb = tf.keras.layers.Embedding(tgt_vocab, d_model)

        self.pos_enc_src = self._positional_encoding(max_src, d_model)
        self.pos_enc_tgt = self._positional_encoding(max_tgt, d_model)

        # Encoder
        self.enc_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.enc_norm1 = tf.keras.layers.LayerNormalization()
        self.enc_ffn = tf.keras.Sequential([tf.keras.layers.Dense(dff, activation='relu'),
                                            tf.keras.layers.Dense(d_model)])
        self.enc_norm2 = tf.keras.layers.LayerNormalization()

        # Decoder
        self.dec_self_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.dec_norm1 = tf.keras.layers.LayerNormalization()
        self.dec_cross_attn = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.dec_norm2 = tf.keras.layers.LayerNormalization()
        self.dec_ffn = tf.keras.Sequential([tf.keras.layers.Dense(dff, activation='relu'),
                                            tf.keras.layers.Dense(d_model)])
        self.dec_norm3 = tf.keras.layers.LayerNormalization()

        self.drop = tf.keras.layers.Dropout(dropout)
        self.out = tf.keras.layers.Dense(tgt_vocab)

    def _positional_encoding(self, max_len, dm):
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(dm)[np.newaxis, :]
        rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(dm))
        rads = pos * rates
        rads[:, 0::2] = np.sin(rads[:, 0::2])
        rads[:, 1::2] = np.cos(rads[:, 1::2])
        return tf.cast(rads[np.newaxis, :, :], tf.float32)

    def call(self, inputs, training=False):
        src, tgt = inputs
        scale = tf.math.sqrt(tf.cast(self.d_model, tf.float32))

        # Encoder
        e = self.enc_emb(src) * scale + self.pos_enc_src[:, :tf.shape(src)[1], :]
        e = self.drop(e, training=training)
        a = self.enc_attn(e, e, e, training=training)
        e = self.enc_norm1(e + a)
        e = self.enc_norm2(e + self.enc_ffn(e))

        # Decoder with causal mask
        seq_len = tf.shape(tgt)[1]
        causal = tf.cast(tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0), tf.bool)

        d = self.dec_emb(tgt) * scale + self.pos_enc_tgt[:, :seq_len, :]
        d = self.drop(d, training=training)
        d = self.dec_norm1(d + self.dec_self_attn(d, d, d, attention_mask=causal, training=training))
        d = self.dec_norm2(d + self.dec_cross_attn(query=d, value=e, key=e, training=training))
        d = self.dec_norm3(d + self.dec_ffn(d))

        return self.out(d)

model = SummarizationTransformer(
    src_vocab=doc_vocab_size, tgt_vocab=sum_vocab_size,
    d_model=128, num_heads=4, dff=256,
    max_src=MAX_DOC_LEN, max_tgt=MAX_SUM_LEN
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

print("Model compiled.")

In [ ]:
# Train
dec_input = sum_tensor[:, :-1]
dec_target = sum_tensor[:, 1:]

print(f"Training on {len(doc_tensor)} examples...")
print(f"Encoder input: {doc_tensor.shape}, Decoder input: {dec_input.shape}, Target: {dec_target.shape}")

history = model.fit(
    [doc_tensor, dec_input], dec_target,
    epochs=30, batch_size=64, validation_split=0.1,
    verbose=1
)

In [ ]:
# Plot training
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history.history['accuracy'], label='Train')
ax2.plot(history.history['val_accuracy'], label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Inference — autoregressive decoding

def summarize(text, max_len=MAX_SUM_LEN - 1):
    seq = doc_tokenizer.texts_to_sequences([text[:300]])
    enc_input = pad_sequences(seq, maxlen=MAX_DOC_LEN, padding='post', truncating='post')

    sos_id = sum_tokenizer.word_index['<sos>']
    eos_id = sum_tokenizer.word_index.get('<eos>', -1)

    dec_in = np.zeros((1, max_len), dtype=np.int32)
    dec_in[0, 0] = sos_id

    result = []
    for i in range(1, max_len):
        preds = model([enc_input, dec_in], training=False)
        next_id = tf.argmax(preds[0, i - 1], axis=-1).numpy()
        dec_in[0, i] = next_id
        if next_id == eos_id:
            break
        word = sum_tokenizer.index_word.get(next_id, '')
        if word and word not in ('<OOV>', '<sos>'):
            result.append(word)
    return ' '.join(result)

# Test on a few training examples
print("Sample summaries from the trained model:\n")
for idx in [0, 5, 10, 50, 100]:
    if idx >= len(documents):
        break
    pred = summarize(documents[idx])
    ref = summaries[idx].replace('<sos>', '').replace('<eos>', '').strip()
    print(f"DOC:  {documents[idx][:120]}...")
    print(f"REF:  {ref}")
    print(f"PRED: {pred}")
    print()

In [ ]:
# Quantitative evaluation on first 50 examples

smoother = SmoothingFunction().method1
rouge_sc = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
eval_results = []

for i in range(min(50, len(documents))):
    ref = summaries[i].replace('<sos>', '').replace('<eos>', '').strip()
    pred = summarize(documents[i])

    ref_tok = [nltk.word_tokenize(ref)]
    pred_tok = nltk.word_tokenize(pred) if pred else ['<empty>']

    bleu = sentence_bleu(ref_tok, pred_tok, smoothing_function=smoother)
    rouge = rouge_sc.score(ref, pred if pred else '<empty>')

    eval_results.append({
        'BLEU': bleu,
        'ROUGE-1': rouge['rouge1'].fmeasure,
        'ROUGE-2': rouge['rouge2'].fmeasure,
        'ROUGE-L': rouge['rougeL'].fmeasure
    })

eval_df = pd.DataFrame(eval_results)
print("Average scores over 50 examples:")
print(eval_df.mean().round(4).to_string())
print(f"\nNote: With only 2K training examples and a single-layer Transformer,")
print(f"these scores are expected to be low. The architecture & pipeline are correct.")
print(f"Scaling to 100K+ examples with 6 layers would yield competitive results.")